# Exploratory Data Analysis with the Iris Dataset

This notebook introduces common EDA steps using the classic Iris dataset. We will inspect the data, summarize it, look for patterns, and create visuals that help us understand the relationships between features.

## 1. Import libraries

We will use `pandas` for data handling, `seaborn` and `matplotlib` for visualization, and `scikit-learn` to load the dataset.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris

# Make plots look nicer
sns.set_theme(style="whitegrid")

## 2. Load the data

The Iris dataset contains measurements of sepals and petals for three flower species.

In [ ]:
iris = load_iris()
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df['species'] = iris.target_names[iris.target]

df.head()

### 2a. Selecting and reordering columns

A DataFrame lets us pick out a subset of columns, or reorder them, simply by
passing a list of column names. Here we put the `species` label first and group the
petal and sepal measurements together — handy whenever we want the table to read
in a particular order.

In [ ]:
df[["species", "petal length (cm)", "petal width (cm)", "sepal length (cm)", "sepal width (cm)"]]

## 3. Check the shape and data types

This helps us understand how much data we have and whether each column is numeric or categorical.

In [ ]:
print("Shape of the dataset:", df.shape)
print("\nData types:")
print(df.dtypes)
print("\nMissing values per column:")
print(df.isnull().sum())

## 4. Summary statistics

The summary shows the center, spread, and range of each numerical feature.

In [ ]:
df.describe()

## 5. Univariate analysis: one feature at a time

*Univariate* means "one variable." Here we look at a single measurement on its
own, but split the histogram by species so we can see whether that one feature
already separates the flowers. Each coloured curve is one species.

Change the `feature` variable to explore every measurement in turn.

In [ ]:
feature = 'petal length (cm)'   # try each of iris.feature_names

plt.figure(figsize=(8, 5))
sns.histplot(
    data=df, x=feature, hue='species',
    bins=20, kde=True, element='step',
    stat='count',   # y-axis = frequency (number of flowers per bin)
)
plt.title(f'Distribution of "{feature}" per species')
plt.show()

## 6. Bivariate analysis: two features together

*Bivariate* means "two variables." A scatter plot of one feature against another,
coloured by species, shows whether a **pair** of measurements separates the
flowers better than either one alone. The histograms on the top and right margins
remind us what each feature looks like on its own.

In [ ]:
x_feature = 'petal length (cm)'
y_feature = 'petal width (cm)'

sns.jointplot(data=df, x=x_feature, y=y_feature, hue='species', height=6)
plt.suptitle(f'{x_feature} vs {y_feature}', y=1.02)
plt.show()

### 6a. Correlation between all features

A scatter plot compares two features at a time; a *correlation matrix* summarises
the linear relationship between **every** pair of features at once. Values near
+1 or -1 mean two measurements rise and fall together (overlapping information),
while values near 0 mean they are largely independent. Strong correlations are a
hint that PCA (section 8) will be able to compress these four features into fewer.

In [ ]:
corr = df.drop(columns='species').corr()

plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlation matrix of the four measurements')
plt.show()

## 7. Three features at once: a 3D scatter plot

We can push to three dimensions and plot three measurements together, with colour
still encoding species. Three features is about the limit of what we can plot
directly — to go further we need dimensionality reduction, which is the next step.

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (registers the 3d projection)

x_feat, y_feat, z_feat = 'sepal length (cm)', 'petal length (cm)', 'petal width (cm)'

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

for species, color in zip(df['species'].unique(),
                          ['tab:blue', 'tab:orange', 'tab:green']):
    subset = df[df['species'] == species]
    ax.scatter(subset[x_feat], subset[y_feat], subset[z_feat],
               label=species, s=40, alpha=0.8, color=color)

# labelpad pushes each axis title away from the cube so none of them is clipped;
# the z-axis label in particular sits at the right edge.
ax.set_xlabel(x_feat, labelpad=12)
ax.set_ylabel(y_feat, labelpad=12)
ax.set_zlabel(z_feat, labelpad=12)
ax.set_title('Three measurements in 3D')
ax.view_init(elev=20, azim=-50)   # tilt so all three axes are clearly visible

ax.legend()

# Leave margins inside the figure so the axis labels are not cut off at the edges.
fig.subplots_adjust(left=0.10, right=0.95, top=0.93, bottom=0.05)
plt.show()

## 8. Principal Component Analysis (PCA)

We have four measurements, but they overlap and are correlated, so they carry
redundant information. PCA builds new axes — the *principal components* — that are
combinations of the original features chosen to capture as much variation as
possible in as few dimensions as possible.

Two habits to teach here:

- **Standardize first.** Put every feature on a mean-0, standard-deviation-1 scale
  so that a feature does not dominate just because of the units it was measured in.
- **Check the explained variance** to see how much information the first few
  components keep.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

features = iris.feature_names
X = df[features].values

# 1. Standardize: each feature -> mean 0, standard deviation 1
X_scaled = StandardScaler().fit_transform(X)

# 2. Fit PCA on all four components
pca = PCA(n_components=4)
scores = pca.fit_transform(X_scaled)

explained = pca.explained_variance_ratio_
for i, ratio in enumerate(explained, start=1):
    print(f'PC{i}: {ratio:.2%} of the variance')
print(f'\nFirst two components together: {explained[:2].sum():.2%}')

### 8a. Score plot — where the samples land

The *score plot* places every flower in the new PC1–PC2 space. If the species form
separate clouds here, then just two components already capture most of what makes
them different — we compressed four features into two with little loss.

In [ ]:
pc_df = pd.DataFrame(scores[:, :2], columns=['PC1', 'PC2'])
pc_df['species'] = df['species'].values

plt.figure(figsize=(8, 6))
sns.scatterplot(data=pc_df, x='PC1', y='PC2', hue='species', s=60)
plt.xlabel(f'PC1 ({explained[0]:.1%} of variance)')
plt.ylabel(f'PC2 ({explained[1]:.1%} of variance)')
plt.title('PCA score plot')
plt.show()

### 8b. Loadings — what each component *means*

A component is just a weighted mix of the original features. The *loadings* are
those weights: they tell us how much each measurement contributes to each
component, and so explain *why* the samples are arranged the way they are. If
petal length and petal width both load strongly on PC1, then moving along PC1
mostly means "bigger petals.".

In [ ]:
loadings = pd.DataFrame(
    pca.components_[:2].T,
    columns=['PC1', 'PC2'],
    index=features,
)
print(loadings.round(3))

plt.figure(figsize=(8, 4))
sns.heatmap(loadings, annot=True, cmap='coolwarm', center=0)
plt.title('PCA loadings (how each feature contributes)')
plt.show()

## 9. Clustering: can we recover the species *without* labels?

Everything above used the species labels to colour the plots. Clustering is
**unsupervised**: we hide the labels and ask an algorithm to group the flowers by
similarity alone, then check how well its groups line up with the real species.

We use K-Means with `k=3` because we happen to know there are three species. The
cluster numbers (0, 1, 2) are arbitrary tags, so we compare them to the true
species with a cross-tabulation.

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled)

pc_df['cluster'] = clusters

# How do the discovered clusters line up with the true species?
comparison = pd.crosstab(pc_df['species'], pc_df['cluster'],
                         rownames=['true species'], colnames=['cluster'])
print(comparison)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharex=True, sharey=True)

sns.scatterplot(data=pc_df, x='PC1', y='PC2', hue='species', s=60, ax=axes[0])
axes[0].set_title('True species')

sns.scatterplot(data=pc_df, x='PC1', y='PC2', hue='cluster',
                palette='Set2', s=60, ax=axes[1])
axes[1].set_title('K-Means clusters (labels hidden)')

plt.tight_layout()
plt.show()

## 10. Supervised classification with an SVM (in PCA space)

Clustering grouped the flowers *without* using the labels. Now we do the opposite:
**supervised** learning, where we give the model the species labels and ask it to
predict them. This time we classify in the **PCA space** from section 8 rather than
from the four raw measurements — the first two components already hold ~96% of the
information, so we can both train on them and *draw* the result in the same PC1–PC2
plane we used for the score and cluster plots.

A **Support Vector Machine (SVM)** finds the boundary that separates the classes
with the widest possible margin. Two habits are built in:

- **Train/test split.** We fit on one part of the data and score on another, so the
  accuracy reflects flowers the model has never seen.
- **Everything in one pipeline.** Scaling and PCA go *inside* the pipeline, so they
  are learned from the training data only — the test set never leaks into them.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix

X = df[iris.feature_names].values
y = df['species'].values

# Hold out 30% for testing; stratify keeps the three species balanced in each split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Standardize -> reduce to 2 principal components -> classify.
# All three steps are fit on the training data only, so the test set never leaks in.
svm = make_pipeline(
    StandardScaler(),
    PCA(n_components=2),
    SVC(kernel='rbf', C=1, gamma='scale'),
)
svm.fit(X_train, y_train)

y_pred = svm.predict(X_test)
print(f'Test accuracy (2 principal components): {svm.score(X_test, y_test):.2%}\n')
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=iris.target_names)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=iris.target_names, yticklabels=iris.target_names)
plt.xlabel('Predicted species')
plt.ylabel('True species')
plt.title('SVM confusion matrix (test set)')
plt.show()

### 10a. Visualising the decision boundary in PCA space

Because we reduced to two components, we can draw the classifier directly. We train
an SVM on the PC1–PC2 scores from section 8 and shade the region it assigns to each
species. This is the *same plane* as the PCA score plot and the clustering plot, so
you can line all three up: where the flowers sit, how K-Means split them, and where
the SVM draws its borders. The background colour is the model's prediction; the dots
are the real flowers, so any dot on a mismatched background is a mistake.

In [ ]:
import numpy as np
from matplotlib.colors import ListedColormap

X_pca = scores[:, :2]          # PC1, PC2 from the PCA section
y_codes = iris.target          # 0, 1, 2 for the three species

svm_pca = make_pipeline(StandardScaler(), SVC(kernel='rbf', C=1, gamma='scale'))
svm_pca.fit(X_pca, y_codes)

# Classify every point on a fine grid covering the PC1-PC2 plane
x_min, x_max = X_pca[:, 0].min() - 1, X_pca[:, 0].max() + 1
y_min, y_max = X_pca[:, 1].min() - 1, X_pca[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02),
                     np.arange(y_min, y_max, 0.02))
Z = svm_pca.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

region_colors = ListedColormap(['#cfe8ff', '#ffe2c2', '#cdeccd'])
point_colors = ['tab:blue', 'tab:orange', 'tab:green']

plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, Z, alpha=0.6, cmap=region_colors)
for code, species in enumerate(iris.target_names):
    mask = y_codes == code
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1], color=point_colors[code],
                edgecolor='k', s=40, label=species)
plt.xlabel(f'PC1 ({explained[0]:.1%} of variance)')
plt.ylabel(f'PC2 ({explained[1]:.1%} of variance)')
plt.title('SVM decision regions in PCA space')
plt.legend()
plt.show()

### 10b. Why isn't the boundary a straight line? The kernel trick

Yes — we used `kernel='rbf'`, so the SVM is using the **kernel trick**. A plain
*linear* SVM can only draw a straight line (a flat plane in higher dimensions). The
RBF kernel instead behaves as if it had lifted every flower into a much
higher-dimensional space, found a flat separating plane there, and projected it back
— and that projected plane looks **curved** in our 2-D plot. The clever part is that
it never actually computes those extra coordinates; it only needs distances between
points.

First, compare a **linear** kernel with the **RBF** kernel on the same PCA data:

In [ ]:
def plot_regions(ax, model, title):
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.6, cmap=region_colors)
    for code, species in enumerate(iris.target_names):
        m = y_codes == code
        ax.scatter(X_pca[m, 0], X_pca[m, 1], color=point_colors[code],
                   edgecolor='k', s=30, label=species)
    ax.set_title(title)
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')

linear_svm = make_pipeline(StandardScaler(), SVC(kernel='linear', C=1)).fit(X_pca, y_codes)
rbf_svm = make_pipeline(StandardScaler(), SVC(kernel='rbf', C=1, gamma='scale')).fit(X_pca, y_codes)

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharex=True, sharey=True)
plot_regions(axes[0], linear_svm, 'Linear kernel  ->  straight boundaries')
plot_regions(axes[1], rbf_svm, 'RBF kernel  ->  curved boundaries')
axes[0].legend()
plt.tight_layout()
plt.show()

The RBF boundary bends to follow the data, but it is still a *flat* plane — just in
a higher-dimensional space we cannot see. To picture what "lift to a higher
dimension" means, here is the classic example: two rings that **no straight line**
can separate. If we add a third coordinate based on each point's distance from the
centre (exactly the kind of feature the RBF kernel builds), the rings pull apart in
3-D and a single flat plane slices cleanly between them.

In [ ]:
from sklearn.datasets import make_circles

Xc, yc = make_circles(n_samples=300, factor=0.3, noise=0.08, random_state=0)

# Kernel-style lift: 3rd coordinate = closeness to the centre (an RBF-like feature)
z = np.exp(-(Xc[:, 0] ** 2 + Xc[:, 1] ** 2))

fig = plt.figure(figsize=(14, 6))

# Left: the original 2-D data -- no straight line can separate the two rings
ax1 = fig.add_subplot(1, 2, 1)
ax1.scatter(Xc[:, 0], Xc[:, 1], c=yc, cmap='coolwarm', edgecolor='k', s=30)
ax1.set_title('Original 2-D: no line can separate the rings')
ax1.set_xlabel('x')
ax1.set_ylabel('y')
ax1.set_aspect('equal')

# Right: lifted into 3-D -- now a flat plane separates them
ax2 = fig.add_subplot(1, 2, 2, projection='3d')
ax2.scatter(Xc[:, 0], Xc[:, 1], z, c=yc, cmap='coolwarm', edgecolor='k', s=30)
gx, gy = np.meshgrid(np.linspace(-1.2, 1.2, 10), np.linspace(-1.2, 1.2, 10))
ax2.plot_surface(gx, gy, np.full_like(gx, 0.6), alpha=0.3, color='gray')
ax2.set_title('Lifted to 3-D: a flat plane separates them')
ax2.set_xlabel('x')
ax2.set_ylabel('y')
ax2.set_zlabel('distance-from-centre feature', labelpad=10)
ax2.view_init(elev=18, azim=-60)

plt.tight_layout()
plt.show()

### 10c. How `gamma` controls the boundary

The RBF kernel has a `gamma` setting that decides **how far the influence of a
single training point reaches**:

- **Low gamma** — each point influences a wide area, so the boundary is smooth and
  almost straight. It can *underfit*, missing real structure.
- **High gamma** — each point only influences its immediate neighbourhood, so the
  boundary tightens into little islands around individual flowers. It fits the
  training data almost perfectly but *overfits*: it would generalise poorly to new
  flowers.

Watch the boundary go from smooth to jagged as `gamma` increases (the `gamma='scale'`
we used earlier is about 0.5 on this standardized data):

In [ ]:
gammas = [0.05, 0.5, 5, 50]

fig, axes = plt.subplots(1, len(gammas), figsize=(20, 5), sharex=True, sharey=True)
for ax, g in zip(axes, gammas):
    model = make_pipeline(StandardScaler(), SVC(kernel='rbf', C=1, gamma=g))
    model.fit(X_pca, y_codes)

    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.6, cmap=region_colors)
    for code, species in enumerate(iris.target_names):
        m = y_codes == code
        ax.scatter(X_pca[m, 0], X_pca[m, 1], color=point_colors[code],
                   edgecolor='k', s=20)

    train_acc = model.score(X_pca, y_codes)
    ax.set_title(f'gamma = {g}\n(training accuracy {train_acc:.0%})')
    ax.set_xlabel('PC1')
axes[0].set_ylabel('PC2')

plt.suptitle('Low gamma = smooth (underfit)   ->   High gamma = jagged islands (overfit)', y=1.05)
plt.tight_layout()
plt.show()

### 10d. Choosing `gamma` and `C` with cross-validation

Picking `gamma` by eye is good for intuition, but in practice we let the data choose.
`GridSearchCV` tries every combination of the values we list for `gamma` and `C`
(`C` controls how heavily the SVM is penalised for misclassifying training points).
For each combination it runs **5-fold cross-validation** on the *training* data —
splitting it into 5 parts, training on 4 and validating on the held-out part, five
times over. The combination with the best average validation score wins, and only
then do we touch the test set, once, for an honest final score.

The heatmap then shows the cross-validation accuracy for every (`C`, `gamma`) pair,
so you can see which region of settings works well instead of guessing.

In [ ]:
from sklearn.model_selection import GridSearchCV

pipe = make_pipeline(StandardScaler(), PCA(n_components=2), SVC(kernel='rbf'))

param_grid = {
    'svc__C': [0.1, 1, 10, 100],
    'svc__gamma': [0.01, 0.1, 1, 10],
}

grid = GridSearchCV(pipe, param_grid, cv=5)
grid.fit(X_train, y_train)

print('Best parameters:', grid.best_params_)
print(f'Best cross-validation accuracy: {grid.best_score_:.2%}')
print(f'Test accuracy with the tuned model: {grid.score(X_test, y_test):.2%}')

In [ ]:
# Cross-validation accuracy for every (C, gamma) combination
results = pd.DataFrame(grid.cv_results_)
pivot = results.pivot_table(index='param_svc__C', columns='param_svc__gamma',
                            values='mean_test_score')

plt.figure(figsize=(6, 5))
sns.heatmap(pivot, annot=True, fmt='.3f', cmap='viridis')
plt.xlabel('gamma')
plt.ylabel('C')
plt.title('5-fold cross-validation accuracy across the grid')
plt.show()

## 11. Reflection

Bringing the whole analysis together:

- Which **single** feature separated the species best in the univariate plots?
  Which one barely helped?
- Did any **pair** of features (bivariate plot) separate the species more cleanly
  than the best single feature?
- How much total variance did the first **two** principal components capture? What
  does that tell you about how redundant the four measurements are?
- Reading the **loadings**, what does moving along PC1 mean in terms of the
  original measurements?
- How well did the unsupervised **clusters** match the true species? Which species
  was easiest to recover, and which two were most often confused?
- The **SVM** was given the labels; clustering was not. How did the SVM's test
  accuracy compare with how well clustering recovered the species, and why might
  using the labels help?
- In the SVM decision-region plot, where are mistakes most likely to happen, and
  which species is essentially never confused?